Agents and Tools - ToolNodes

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
import requests
from langchain_groq import ChatGroq
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition, ToolNode


class AgentState(BaseModel):
    messages: Annotated[list, add_messages]


@tool
def devpost_fetcher(query: Annotated [str, "Search term for hackathons on Devpost"] )-> str:
    """
    Fetches upcoming hackathons from Devpost.

    Use when the user asks about Devpost events, hackathon deadlines,
    themes, prizes, or eligibility on Devpost.

    Args:
        query: Search term or filter, e.g. "AI hackathons in India"
               or "web3 events with deadline before December 2026".
    """
    url =  "https://devpost.com/api/hackathons"
    params = {
        "challenge_type[0]": "online",   
        "search_query": query,
        "order_by": "newest",
        "page": 1,
        "per_page": 20,
    }

    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as e:
        return f"Devpost fetch Failed: {e}"




    hackathons = data.get("hackathons", [])
    if not hackathons:
        return "No hackathons found on Devpost."

    lines = []
    for h in hackathons:
        lines.append(
            f"• {h['title']}\n"
            f"  URL: {h['url']}\n"
            f"  Deadline: {h.get('time_left_to_submission', 'N/A')}\n"
            f"  Prize: {h.get('prize_amount', 'N/A')}\n"
            f"  Theme: {', '.join(h.get('themes', []))}\n"
            f"  Registrations: {h.get('registrations_count', 'N/A')}\n"
            f"  Status: {h.get('open_state', 'unknown')}"
        )
    return "\n\n".join(lines)
            
        
    

@tool
def devfolio_fetcher(query: Annotated[str, "Search term for hackathons on Devfolio"] ) -> str:
    """
    Fetches upcoming hackathons from Devfolio.

    Use when the user asks about Devfolio events, hackathon deadlines,
    themes, prizes, or eligibility on Devfolio.

    Args:
        query: Search term or filter, e.g. "AI/ML hackathons"
               or "online events open to students".
    """
    # your requests.get / scraping logic here
    ...


tools = [devfolio_fetcher, devpost_fetcher]


llm = ChatGroq(model="openai/gpt-oss-20b").bind_tools(tools)


def call_model(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def call_tools(state:AgentState):
    result = ToolNode(tools).invoke(state)
    return result




graph = StateGraph(AgentState)
graph.add_node("agents", call_model)
graph.add_node("tools", call_tools)


graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")


agent = graph.compile()


result = agent.invoke({
    "messages": [HumanMessage("Find Hackathons which has deadlines in November")]
})
